# Control Task Results

This notebook summarizes the named-entity follow-up analyses and removes the old `by -> near` control-task comparison. It focuses on three questions:

- How much named-entity circuit behavior degrades when we remove the top 100 edges versus the whole circuit.
- How well the original animacy circuits transfer to the BLiMP passive-prefix control with our target sets.
- How the named-entity discovery runs compare to the original animacy circuits across models.


In [6]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd
from plotly.subplots import make_subplots

try:
    from IPython.display import display
except ImportError:
    def display(value):
        return value

MODEL_ALIASES = {
    'gpt2': 'gpt2',
    'gpt-2': 'gpt2',
    'llama-3.2-3b': 'meta-llama/Llama-3.2-3B',
    'meta-llama/llama-3.2-3b': 'meta-llama/Llama-3.2-3B',
    'gemma-3-4b-pt': 'google/gemma-3-4b-pt',
    'google/gemma-3-4b-pt': 'google/gemma-3-4b-pt',
    'qwen-3-4b': 'Qwen/Qwen3-4B',
    'qwen/qwen3-4b': 'Qwen/Qwen3-4B',
}


def normalize_model_alias_key(model_name: str) -> str:
    return re.sub(r'[\s_]+', '-', model_name.strip().casefold())


def canonical_model_name(model_name: str) -> str:
    stripped = model_name.strip()
    return MODEL_ALIASES.get(normalize_model_alias_key(stripped), stripped)


def safe_model_name(model_name: str) -> str:
    slug = re.sub(r'[^A-Za-z0-9_.-]+', '_', model_name).strip('_')
    return slug or 'model'


def resolve_animacy_circuit_root(start: Path | str | None = None) -> Path:
    anchor = Path(start) if start is not None else Path.cwd()
    anchor = anchor.resolve()
    if anchor.is_file():
        anchor = anchor.parent

    for base in (anchor, *anchor.parents):
        for candidate in (base, base / 'animacy-circuit'):
            if (
                candidate.is_dir()
                and (candidate / 'dataset').is_dir()
                and (candidate / 'results').is_dir()
                and (candidate / 'scripts').is_dir()
            ):
                return candidate

    raise FileNotFoundError('Could not locate the animacy-circuit root.')


NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = resolve_animacy_circuit_root(NOTEBOOK_DIR)
RESULTS_ROOT = REPO_ROOT / 'results'
LOCALIZATION_ROOT = RESULTS_ROOT / 'eap_ig_localization'
NAMED_ENTITY_ROOT = RESULTS_ROOT / 'named_entity_discovery'

MODEL_NAMES = (
    'gpt2',
    'meta-llama/Llama-3.2-3B',
    'google/gemma-3-4b-pt',
    'Qwen/Qwen3-4B',
)

CIRCUIT_VARIANT_ORDER = [
    'Original 85% transfer',
    'Named 85%',
    'Named best',
    'Named @ original 85% size',
]

SOURCE_REFERENCE_ORDER = [
    'Original 85%',
    'Original best',
]

CIRCUIT_VARIANT_COLORS = {
    'Original 85% transfer': '#1f77b4',
    'Named 85%': '#f58518',
    'Named best': '#54a24b',
    'Named @ original 85% size': '#e45756',
}

COMPARISON_LEGEND_LABELS = {
    'Original 85% transfer': 'Original circuit',
    'Named 85%': 'Named circuit',
}

pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 4)


In [7]:
def _latest_path(candidates: list[Path]) -> Path:
    if not candidates:
        raise FileNotFoundError('No matching artifact found.')
    return max(candidates, key=lambda path: (path.parts, path.stat().st_mtime))


def _resolve_repo_relative(path_str: str) -> Path:
    path = Path(path_str)
    if path.is_absolute():
        return path
    if path.parts and path.parts[0] == REPO_ROOT.name:
        return REPO_ROOT.parent / path
    return REPO_ROOT / path


def _best_budget_row(frame: pd.DataFrame) -> pd.Series:
    numeric = frame.copy()
    ranked = numeric.sort_values(
        ['faithfulness_mean', 'accuracy_mean', 'induced_node_count', 'collapsed_edge_budget'],
        ascending=[False, False, True, True],
    )
    return ranked.iloc[0]


def _budget_row(frame: pd.DataFrame, budget: int) -> pd.Series:
    exact = frame.loc[frame['collapsed_edge_budget'] == budget]
    if not exact.empty:
        return exact.iloc[0]
    closest_index = (frame['collapsed_edge_budget'] - budget).abs().idxmin()
    return frame.loc[closest_index]


def _edge_prefixes(edge_path: Path, budget: int) -> set[str]:
    frame = pd.read_csv(edge_path)
    if 'collapsed_edge' in frame.columns:
        values = frame['collapsed_edge'].astype(str).tolist()[:budget]
        return set(values)
    column = 'edge' if 'edge' in frame.columns else 'edge_name'
    values = frame[column].astype(str).tolist()[:budget]
    return {value.split('<', maxsplit=1)[0] for value in values}


def _iou(left: set[str], right: set[str]) -> float:
    union = left | right
    if not union:
        return 0.0
    return len(left & right) / len(union)


def find_named_entity_localization_csv(model_name: str) -> Path:
    model_dir = LOCALIZATION_ROOT / safe_model_name(model_name)
    direct = sorted(model_dir.glob('named_entity_truncated_localization_*/topk_evaluations.csv'))
    if direct:
        return _latest_path(direct)
    partial = sorted(model_dir.glob('named_entity_truncated_localization_*/sample_*/seed_*/topk_evaluations_partial_*.csv'))
    return _latest_path(partial)


def find_named_entity_summary_path(model_name: str) -> Path:
    model_dir = NAMED_ENTITY_ROOT / safe_model_name(model_name) / 'model_specific_correct'
    candidates = sorted(model_dir.glob('named_entity_discovery_*/named_entity_discovery_summary_*.json'))
    return _latest_path(candidates)


def find_named_entity_eval_summary_path(model_name: str) -> Path:
    model_dir = RESULTS_ROOT / 'named_entity_circuit_eval' / safe_model_name(model_name) / 'model_specific_correct'
    candidates = sorted(model_dir.glob('named_entity_circuit_eval_*/named_entity_85pct_circuit_eval_*.json'))
    return _latest_path(candidates)


def find_blimp_summary_path(model_name: str) -> Path:
    model_dir = safe_model_name(model_name)
    candidates = sorted(RESULTS_ROOT.glob(f'*/{model_dir}/controls/blimp_passive_prefix/summary.json'))
    if not candidates:
        raise FileNotFoundError('No BLiMP transfer artifacts found.')
    for candidate in reversed(candidates):
        status_path = candidate.with_name('status.json')
        if not status_path.exists():
            continue
        status = json.loads(status_path.read_text())
        if status.get('selected_budget_row') is not None:
            return candidate
    return _latest_path(candidates)


def find_necessary_summary_path() -> Path:
    candidates = sorted(LOCALIZATION_ROOT.glob('necessary_edge_expansion_main_original_20_50_*/necessary_budget_summary.json'))
    return _latest_path(candidates)


def _edge_components(edges: set[str]) -> set[str]:
    components: set[str] = set()
    for edge in edges:
        parent, child = edge.split('->', maxsplit=1)
        components.add(parent)
        components.add(child)
    return components


def load_localization_summary() -> pd.DataFrame:
    rows = []
    for model_name in MODEL_NAMES:
        csv_path = find_named_entity_localization_csv(model_name)
        df = pd.read_csv(csv_path)
        subset = df[(df['mode'] == 'ablate_top') & (df['baseline'] == 'eap_ranked')].sort_values('collapsed_edge_budget')
        if subset.empty:
            raise ValueError(f"No `ablate_top` / `eap_ranked` rows found in {csv_path}")
        top_100_row = _budget_row(subset, 100)
        whole_circuit_row = subset.iloc[-1]
        rows.append({
            'Model': canonical_model_name(model_name),
            'Top-100 budget used': int(top_100_row['collapsed_edge_budget']),
            'Max circuit faithfulness with top-100 removal': top_100_row['faithfulness_mean'],
            'Max circuit faithfulness with whole-circuit removal': whole_circuit_row['faithfulness_mean'],
            'Max circuit accuracy with Top-100 removal': top_100_row['accuracy_mean'],
            'Max circuit accuracy with whole-circuit removal': whole_circuit_row['accuracy_mean'],
            'Whole-circuit budget': int(whole_circuit_row['collapsed_edge_budget']),
            'Validation examples': int(top_100_row['validation_examples']),
            'Source': str(csv_path.relative_to(REPO_ROOT)),
        })
    return pd.DataFrame(rows).sort_values('Model').reset_index(drop=True)


def load_blimp_transfer_summary() -> pd.DataFrame:
    rows = []
    for model_name in MODEL_NAMES:
        summary_path = find_blimp_summary_path(model_name)
        status_path = summary_path.with_name('status.json')
        summary = json.loads(summary_path.read_text())
        status = json.loads(status_path.read_text())
        selected = status['selected_budget_row']
        rows.append({
            'Model': canonical_model_name(model_name),
            'Examples': int(summary['example_count']),
            'Full-model accuracy': summary['full_model_accuracy'],
            'Circuit accuracy': summary['circuit_accuracy'],
            'Accuracy delta (circuit - full)': summary['accuracy_delta_circuit_minus_full'],
            'Circuit fix rate': summary['circuit_fix_rate'],
            'Circuit break rate': summary['circuit_break_rate'],
            'Selected budget': int(status['selected_budget']),
            'Selected circuit faithfulness': selected['faithfulness_mean'],
            'Selected circuit accuracy': selected['accuracy_mean'],
            'Dataset source': str(_resolve_repo_relative(status['dataset_source']).relative_to(REPO_ROOT)),
            'Source': str(summary_path.relative_to(REPO_ROOT)),
        })
    return pd.DataFrame(rows).sort_values('Model').reset_index(drop=True)


def load_named_entity_variant_summary() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    reference_rows = []
    variant_rows = []
    overlap_rows = []
    target_rows = []
    for model_name in MODEL_NAMES:
        summary_path = find_named_entity_summary_path(model_name)
        eval_summary_path = find_named_entity_eval_summary_path(model_name)
        summary = json.loads(summary_path.read_text())
        eval_summary = json.loads(eval_summary_path.read_text())
        model_label = canonical_model_name(model_name)
        named_first = summary['budget_sweep']['first_threshold_row']
        named_best = summary['budget_sweep']['best_budget_row']
        original_source = summary['overlap']['original_source']
        named_source = summary['overlap']['named_entity_source']
        original_budget_frame = pd.read_csv(_resolve_repo_relative(original_source['budget_path']))
        named_budget_frame = pd.read_csv(_resolve_repo_relative(named_source['budget_path']))
        original_first = original_source['first_threshold_row']
        original_best = _best_budget_row(original_budget_frame)
        original_budget = int(original_first['collapsed_edge_budget'])
        named_same_size = _budget_row(named_budget_frame, original_budget)
        original_transfer = eval_summary['circuit_eval']

        target_rows.append({
            'Model': model_label,
            'Animate targets': int(summary['target_counts']['animate']),
            'Inanimate targets': int(summary['target_counts']['inanimate']),
            'Scored truncated examples': int(summary['dataset_counts']['truncated_scored']),
            'Model-success examples': int(summary['dataset_counts']['model_success']),
            'Discovery examples': int(summary['dataset_counts']['discovery']),
            'Validation examples': int(summary['dataset_counts']['validation']),
        })

        reference_rows.append({
            'Model': model_label,
            'Circuit': 'Original 85%',
            'Evaluation set': 'Original animacy source sweep',
            'Budget': int(original_first['collapsed_edge_budget']),
            'Budget (% of model)': 100.0 * float(original_first['budget_fraction']),
            'Faithfulness': float(original_first['faithfulness_mean']),
            'Accuracy': float(original_first['accuracy_mean']),
            'Validation examples': int(original_first['validation_examples']),
        })
        reference_rows.append({
            'Model': model_label,
            'Circuit': 'Original best',
            'Evaluation set': 'Original animacy source sweep',
            'Budget': int(original_best['collapsed_edge_budget']),
            'Budget (% of model)': 100.0 * float(original_best['budget_fraction']),
            'Faithfulness': float(original_best['faithfulness_mean']),
            'Accuracy': float(original_best['accuracy_mean']),
            'Validation examples': int(original_best['validation_examples']),
        })

        variant_rows.append({
            'Model': model_label,
            'Circuit': 'Original 85% transfer',
            'Evaluation set': 'Named-entity validation split',
            'Budget': int(eval_summary['evaluated_circuit_budget']),
            'Budget (% of model)': 100.0 * float(original_first['budget_fraction']),
            'Faithfulness': float(original_transfer['faithfulness_mean']),
            'Accuracy': float(original_transfer['accuracy_mean']),
            'Validation examples': int(original_transfer['validation_examples']),
        })
        if named_first is not None:
            variant_rows.append({
                'Model': model_label,
                'Circuit': 'Named 85%',
                'Evaluation set': 'Named-entity validation split',
                'Budget': int(named_first['collapsed_edge_budget']),
                'Budget (% of model)': 100.0 * float(named_first['budget_fraction']),
                'Faithfulness': float(named_first['faithfulness_mean']),
                'Accuracy': float(named_first['accuracy_mean']),
                'Validation examples': int(named_first['validation_examples']),
            })
        variant_rows.append({
            'Model': model_label,
            'Circuit': 'Named best',
            'Evaluation set': 'Named-entity validation split',
            'Budget': int(named_best['collapsed_edge_budget']),
            'Budget (% of model)': 100.0 * float(named_best['budget_fraction']),
            'Faithfulness': float(named_best['faithfulness_mean']),
            'Accuracy': float(named_best['accuracy_mean']),
            'Validation examples': int(named_best['validation_examples']),
        })
        variant_rows.append({
            'Model': model_label,
            'Circuit': 'Named @ original 85% size',
            'Evaluation set': 'Named-entity validation split',
            'Budget': int(named_same_size['collapsed_edge_budget']),
            'Budget (% of model)': 100.0 * float(named_same_size['budget_fraction']),
            'Faithfulness': float(named_same_size['faithfulness_mean']),
            'Accuracy': float(named_same_size['accuracy_mean']),
            'Validation examples': int(named_same_size['validation_examples']),
        })

        original_edge_path = _resolve_repo_relative(original_source['edge_path'])
        named_edge_path = _resolve_repo_relative(named_source['edge_path'])
        original_85_edges = _edge_prefixes(original_edge_path, int(original_first['collapsed_edge_budget']))
        original_best_edges = _edge_prefixes(original_edge_path, int(original_best['collapsed_edge_budget']))
        overlap_rows.append({
            'Model': model_label,
            'IoU: Original 85% vs Named 85%': None if named_first is None else _iou(original_85_edges, _edge_prefixes(named_edge_path, int(named_first['collapsed_edge_budget']))),
            'IoU: Original 85% vs Named best': _iou(original_85_edges, _edge_prefixes(named_edge_path, int(named_best['collapsed_edge_budget']))),
            'IoU: Original 85% vs Named @ original 85% size': _iou(original_85_edges, _edge_prefixes(named_edge_path, int(named_same_size['collapsed_edge_budget']))),
            'IoU: Original 85% vs Original best': _iou(original_85_edges, original_best_edges),
        })

    reference_summary = pd.DataFrame(reference_rows)
    reference_summary['Circuit'] = pd.Categorical(reference_summary['Circuit'], SOURCE_REFERENCE_ORDER, ordered=True)
    reference_summary = reference_summary.sort_values(['Model', 'Circuit']).reset_index(drop=True)
    variant_summary = pd.DataFrame(variant_rows)
    variant_summary['Circuit'] = pd.Categorical(variant_summary['Circuit'], CIRCUIT_VARIANT_ORDER, ordered=True)
    variant_summary = variant_summary.sort_values(['Model', 'Circuit']).reset_index(drop=True)
    target_summary = pd.DataFrame(target_rows).sort_values('Model').reset_index(drop=True)
    overlap_summary = pd.DataFrame(overlap_rows).sort_values('Model').reset_index(drop=True)
    return target_summary, reference_summary, variant_summary, overlap_summary


def load_named_entity_necessary_overlap_summary(seed: int = 42, selected_budget: int = 20) -> pd.DataFrame:
    necessary_summary_path = find_necessary_summary_path()
    necessary_summary = json.loads(necessary_summary_path.read_text())
    necessary_edge_frame = pd.read_csv(_resolve_repo_relative(necessary_summary['paths']['collapsed_edges']))
    rows = []
    overlap_order = ['Named 85%', 'Named best', 'Named @ original 85% size']
    for model_name in MODEL_NAMES:
        summary_path = find_named_entity_summary_path(model_name)
        summary = json.loads(summary_path.read_text())
        model_label = canonical_model_name(model_name)
        model_slug = safe_model_name(model_name)
        original_source = summary['overlap']['original_source']
        named_source = summary['overlap']['named_entity_source']
        named_first = summary['budget_sweep']['first_threshold_row']
        named_best = summary['budget_sweep']['best_budget_row']
        original_budget = int(original_source['first_threshold_row']['collapsed_edge_budget'])
        original_85_edges = _edge_prefixes(_resolve_repo_relative(original_source['edge_path']), original_budget)
        named_edge_path = _resolve_repo_relative(named_source['edge_path'])
        named_budget_frame = pd.read_csv(_resolve_repo_relative(named_source['budget_path']))
        named_same_size = _budget_row(named_budget_frame, original_budget)
        necessary_edges = set(
            necessary_edge_frame.loc[
                (necessary_edge_frame['model_slug'] == model_slug)
                & (necessary_edge_frame['seed'] == seed)
                & (necessary_edge_frame['selected_budget'] == selected_budget),
                'collapsed_edge',
            ].astype(str)
        )
        if not necessary_edges:
            raise ValueError(f'No necessary collapsed edges found for {model_slug} seed={seed} budget={selected_budget}.')
        necessary_components = _edge_components(necessary_edges)
        overlap_specs = [
            ('Named 85%', None if named_first is None else int(named_first['collapsed_edge_budget'])),
            ('Named best', int(named_best['collapsed_edge_budget'])),
            ('Named @ original 85% size', int(named_same_size['collapsed_edge_budget'])),
        ]
        for overlap_label, named_budget in overlap_specs:
            overlap_edges = set() if named_budget is None else original_85_edges & _edge_prefixes(named_edge_path, named_budget)
            matched_edges = sorted(necessary_edges & overlap_edges)
            matched_components = sorted(necessary_components & _edge_components(overlap_edges))
            missing_edges = sorted(necessary_edges - overlap_edges)
            rows.append({
                'Model': model_label,
                'Overlap circuit': overlap_label,
                'Necessary collapsed-edge coverage': f'{len(matched_edges)}/{len(necessary_edges)}',
                'Necessary component coverage': f'{len(matched_components)}/{len(necessary_components)}',
                'Missing necessary collapsed edges': ', '.join(missing_edges),
            })
    summary_frame = pd.DataFrame(rows)
    summary_frame['Overlap circuit'] = pd.Categorical(summary_frame['Overlap circuit'], overlap_order, ordered=True)
    return summary_frame.sort_values(['Model', 'Overlap circuit']).reset_index(drop=True)


## Max Circuit Faithfulness / Accuracy Under Removal

These runs use the named-entity truncated localization results and compare two ablations: removing the top 100 ranked edges and removing the whole discovered circuit.


In [8]:
localization_summary = load_localization_summary()
display(localization_summary.round(4))

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=('Faithfulness after removal', 'Accuracy after removal'),
    horizontal_spacing=0.12,
)

fig.add_bar(
    x=localization_summary['Model'],
    y=localization_summary['Max circuit faithfulness with top-100 removal'],
    name='Top-100 removal',
    row=1,
    col=1,
)
fig.add_bar(
    x=localization_summary['Model'],
    y=localization_summary['Max circuit faithfulness with whole-circuit removal'],
    name='Whole-circuit removal',
    row=1,
    col=1,
)
fig.add_bar(
    x=localization_summary['Model'],
    y=localization_summary['Max circuit accuracy with Top-100 removal'],
    name='Top-100 removal',
    showlegend=False,
    row=1,
    col=2,
)
fig.add_bar(
    x=localization_summary['Model'],
    y=localization_summary['Max circuit accuracy with whole-circuit removal'],
    name='Whole-circuit removal',
    showlegend=False,
    row=1,
    col=2,
)

fig.update_layout(barmode="group", height=430, width=1100, title_text="Named-entity circuit removal summary")
fig.update_yaxes(title_text="Faithfulness", row=1, col=1)
fig.update_yaxes(title_text="Accuracy", row=1, col=2)
fig


,Model,Top-100 budget used,Max circuit faithfulness with top-100 removal,Max circuit faithfulness with whole-circuit removal,Max circuit accuracy with Top-100 removal,Max circuit accuracy with whole-circuit removal,Whole-circuit budget,Validation examples,Source
0,Qwen/Qwen3-4B,100,0.0029,0.0002,0.0353,0.0000,103440,878,results/eap_ig_localization/Qwen_Qwen3-4B/name...
1,google/gemma-3-4b-pt,100,0.0121,-0.0005,0.0166,0.0000,5890,3733,results/eap_ig_localization/google_gemma-3-4b-...
2,gpt2,100,-0.0005,0.0001,0.0209,0.0042,2000,1916,results/eap_ig_localization/gpt2/named_entity_...
3,meta-llama/Llama-3.2-3B,100,0.0033,0.0007,0.1009,0.0117,35749,426,results/eap_ig_localization/meta-llama_Llama-3...


## Circuit Transferring on BLiMP With Our Target Sets

This section reads the passive-prefix BLiMP control results and compares the frozen animacy circuits against the full model.


In [9]:
blimp_transfer = load_blimp_transfer_summary()
display(blimp_transfer.round(4))

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=('Full model vs circuit accuracy', 'Fix / break rates'),
    horizontal_spacing=0.12,
)

fig.add_bar(
    x=blimp_transfer['Model'],
    y=blimp_transfer['Full-model accuracy'],
    name='Full model accuracy',
    row=1,
    col=1,
)
fig.add_bar(
    x=blimp_transfer['Model'],
    y=blimp_transfer['Circuit accuracy'],
    name='Circuit accuracy',
    row=1,
    col=1,
)
fig.add_bar(
    x=blimp_transfer['Model'],
    y=blimp_transfer['Circuit fix rate'],
    name='Fix rate',
    row=1,
    col=2,
)
fig.add_bar(
    x=blimp_transfer['Model'],
    y=blimp_transfer['Circuit break rate'],
    name='Break rate',
    row=1,
    col=2,
)

fig.update_layout(barmode="group", height=430, width=1100, title_text="BLiMP passive-prefix transfer")
fig.update_yaxes(title_text="Accuracy", row=1, col=1)
fig.update_yaxes(title_text="Rate", row=1, col=2)
fig


,Model,Examples,Full-model accuracy,Circuit accuracy,Accuracy delta (circuit - full),Circuit fix rate,Circuit break rate,Selected budget,Selected circuit faithfulness,Selected circuit accuracy,Dataset source,Source
0,Qwen/Qwen3-4B,1000,0.844,0.334,-0.510,0.062,0.572,12017,0.8555,0.9761,dataset/blimp/animate_subject_passive.jsonl,results/2026-06-15/Qwen_Qwen3-4B/controls/blim...
1,google/gemma-3-4b-pt,1000,0.920,0.697,-0.223,0.035,0.258,4992,0.8791,0.9726,dataset/blimp/animate_subject_passive.jsonl,results/2026-06-15/google_gemma-3-4b-pt/contro...
2,gpt2,1000,0.854,0.945,0.091,0.143,0.052,1342,0.8621,0.9401,dataset/blimp/animate_subject_passive.jsonl,results/2026-06-15/gpt2/controls/blimp_passive...
3,meta-llama/Llama-3.2-3B,1000,0.886,0.913,0.027,0.100,0.073,16806,0.9209,0.9809,dataset/blimp/animate_subject_passive.jsonl,results/2026-06-15/meta-llama_Llama-3.2-3B/con...


## Named Entities Analysis

This section now separates source-sweep reference numbers from the same-split named-entity comparison.

- `Original source reference` rows are the original animacy metrics on the original animacy validation sweep.
- `Original 85% transfer` is the original 85% circuit evaluated on the named-entity validation split.
- The `Named ...` rows are all evaluated on that same named-entity validation split, so those comparisons are apples-to-apples.


In [10]:
named_entity_targets, named_entity_reference, named_entity_variants, named_entity_overlap = load_named_entity_variant_summary()
named_entity_necessary_overlap = load_named_entity_necessary_overlap_summary()
display(named_entity_targets.round(4))
display(named_entity_reference.round(4))
display(named_entity_variants.round(4))
display(named_entity_overlap.round(4))
display(named_entity_necessary_overlap)

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=('Budget (% of model)', 'Faithfulness', 'Accuracy'),
    horizontal_spacing=0.08,
)

for circuit in CIRCUIT_VARIANT_ORDER:
    subset = named_entity_variants.loc[named_entity_variants['Circuit'] == circuit]
    fig.add_bar(
        x=subset['Model'],
        y=subset['Budget (% of model)'],
        name=circuit,
        legendgroup=circuit,
        marker_color=CIRCUIT_VARIANT_COLORS[circuit],
        row=1,
        col=1,
    )
    fig.add_bar(
        x=subset['Model'],
        y=subset['Faithfulness'],
        name=circuit,
        legendgroup=circuit,
        marker_color=CIRCUIT_VARIANT_COLORS[circuit],
        showlegend=False,
        row=1,
        col=2,
    )
    fig.add_bar(
        x=subset['Model'],
        y=subset['Accuracy'],
        name=circuit,
        legendgroup=circuit,
        marker_color=CIRCUIT_VARIANT_COLORS[circuit],
        showlegend=False,
        row=1,
        col=3,
    )

fig.update_layout(
    barmode="group",
    height=460,
    width=1450,
    title_text="Named-entity validation split: original transfer vs named-entity circuits",
)
fig.update_yaxes(title_text="% of ranked model edges", row=1, col=1)
fig.update_yaxes(title_text="Faithfulness", row=1, col=2)
fig.update_yaxes(title_text="Accuracy", row=1, col=3)
fig


,Model,Animate targets,Inanimate targets,Scored truncated examples,Model-success examples,Discovery examples,Validation examples
0,Qwen/Qwen3-4B,101,107,11093,1378,500,878
1,google/gemma-3-4b-pt,101,107,14437,4233,500,3733
2,gpt2,101,107,10341,2416,500,1916
3,meta-llama/Llama-3.2-3B,101,107,11093,926,500,426


,Model,Circuit,Evaluation set,Budget,Budget (% of model),Faithfulness,Accuracy,Validation examples
0,Qwen/Qwen3-4B,Original 85%,Original animacy source sweep,12017,1.7426,0.8555,0.9761,3800
1,Qwen/Qwen3-4B,Original best,Original animacy source sweep,12017,1.7426,0.8555,0.9761,3800
2,google/gemma-3-4b-pt,Original 85%,Original animacy source sweep,4992,10.7758,0.8791,0.9726,3871
3,google/gemma-3-4b-pt,Original best,Original animacy source sweep,4992,10.7758,0.8791,0.9726,3871
4,gpt2,Original 85%,Original animacy source sweep,1342,11.5580,0.8621,0.9401,4221
5,gpt2,Original best,Original animacy source sweep,2000,17.2250,1.0022,0.9742,4221
6,meta-llama/Llama-3.2-3B,Original 85%,Original animacy source sweep,16806,7.0518,0.9209,0.9809,4234
7,meta-llama/Llama-3.2-3B,Original best,Original animacy source sweep,16806,7.0518,0.9209,0.9809,4234


,Model,Circuit,Evaluation set,Budget,Budget (% of model),Faithfulness,Accuracy,Validation examples
0,Qwen/Qwen3-4B,Original 85% transfer,Named-entity validation split,12017,1.7426,0.5961,0.7927,878
1,Qwen/Qwen3-4B,Named 85%,Named-entity validation split,8836,1.2813,0.9018,0.9237,878
2,Qwen/Qwen3-4B,Named best,Named-entity validation split,16344,2.3701,1.0187,0.9590,878
3,Qwen/Qwen3-4B,Named @ original 85% size,Named-entity validation split,12017,1.7426,0.9719,0.9351,878
4,google/gemma-3-4b-pt,Original 85% transfer,Named-entity validation split,4992,10.7758,0.7912,0.8272,3733
5,google/gemma-3-4b-pt,Named 85%,Named-entity validation split,4992,10.7758,0.8769,0.8679,3733
6,google/gemma-3-4b-pt,Named best,Named-entity validation split,6949,15.0002,1.0157,0.9223,3733
7,google/gemma-3-4b-pt,Named @ original 85% size,Named-entity validation split,4992,10.7758,0.8769,0.8679,3733
8,gpt2,Original 85% transfer,Named-entity validation split,1342,11.5580,0.5262,0.6926,1916
9,gpt2,Named 85%,Named-entity validation split,995,8.5695,0.8858,0.8643,1916


,Model,IoU: Original 85% vs Named 85%,IoU: Original 85% vs Named best,IoU: Original 85% vs Named @ original 85% size,IoU: Original 85% vs Original best
0,Qwen/Qwen3-4B,0.3638,0.3830,0.3891,1.000
1,google/gemma-3-4b-pt,0.6800,0.6355,0.6800,1.000
2,gpt2,0.4164,0.4571,0.4571,0.671
3,meta-llama/Llama-3.2-3B,0.0753,0.2799,0.4516,1.000


In [11]:
comparison_circuits = ['Original 85% transfer', 'Named 85%']
comparison_variants = named_entity_variants.loc[
    named_entity_variants['Circuit'].isin(comparison_circuits)
].copy()

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=('Budget (% of model)', 'Faithfulness', 'Accuracy'),
    horizontal_spacing=0.08,
)

for circuit in comparison_circuits:
    subset = comparison_variants.loc[comparison_variants['Circuit'] == circuit]
    legend_label = COMPARISON_LEGEND_LABELS.get(circuit, circuit)
    fig.add_bar(
        x=subset['Model'],
        y=subset['Budget (% of model)'],
        name=legend_label,
        legendgroup=legend_label,
        marker_color=CIRCUIT_VARIANT_COLORS[circuit],
        row=1,
        col=1,
    )
    fig.add_bar(
        x=subset['Model'],
        y=subset['Faithfulness'],
        name=legend_label,
        legendgroup=legend_label,
        marker_color=CIRCUIT_VARIANT_COLORS[circuit],
        showlegend=False,
        row=1,
        col=2,
    )
    fig.add_bar(
        x=subset['Model'],
        y=subset['Accuracy'],
        name=legend_label,
        legendgroup=legend_label,
        marker_color=CIRCUIT_VARIANT_COLORS[circuit],
        showlegend=False,
        row=1,
        col=3,
    )

fig.update_layout(
    barmode="group",
    height=460,
    width=1450,
    title_text="Named-entity validation split: original vs named 85% circuits",
)
fig.update_yaxes(title_text="% of ranked model edges", row=1, col=1)
fig.update_yaxes(title_text="Faithfulness", row=1, col=2)
fig.update_yaxes(title_text="Accuracy", row=1, col=3)
fig
